# ScopeGuard — Encoding B: Raw Scopes + Offline Flag

**Encoding B** = one binary column per scope (short-named, via
`MultiLabelBinarizer`) **plus a single `offline` flag**. No engineered rubric
features, and no service roll-up: the model gets the bag of scopes and nothing
else. That is the encoding this notebook materializes in Part 1 and trains on
in Part 2.

| Notebook | Encoding |
|---|---|
| `encoding_a` | scope URL columns + 7 engineered rubric features |
| **`encoding_b` (this one)** | short-named scope columns **+ `offline`** |
| `model_pipeline` (C, final) | short-named scope columns **+ per-service columns + `offline`** |

Encoding C is B plus the coarse per-service view. Setting
`INCLUDE_SERVICE_FEATURES = True` in the config below turns this notebook into
C's feature matrix — it is left `False` so that B stays a distinct encoding.

Everything downstream of the feature matrix — Optuna tuning, model selection,
evaluation, SHAP, and every saved figure — is identical to `model_pipeline`,
so the three encodings can be compared like-for-like.

All artifacts are written to **`b_outputs/`**.


## Determinism and imports

Thread-count environment variables are set **before** numpy / XGBoost import,
because the BLAS and OpenMP runtimes read them at load time. Single-threaded
math is what makes results identical on machines with different core counts:
multi-threaded reductions sum in a nondeterministic order.


In [ ]:
import os

# Must come before numpy / sklearn / xgboost are imported.
os.environ["PYTHONHASHSEED"] = "0"
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_var] = "1"

import copy
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from optuna.integration import XGBoostPruningCallback

from sklearn.preprocessing import MultiLabelBinarizer, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, accuracy_score, precision_score, recall_score,
)
from xgboost import XGBClassifier
import shap

import warnings
warnings.filterwarnings("ignore")

optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
# Version fingerprint. Identical seeds only give identical numbers on
# identical library versions, so the run records what it was produced with.
import sklearn, xgboost, matplotlib

ENVIRONMENT = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "shap": shap.__version__,
    "optuna": optuna.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
}
for k, v in ENVIRONMENT.items():
    print(f"{k:>14}: {v}")


## Configuration


In [ ]:
RANDOM_SEED = 42
N_TRIALS = 100          # Optuna trials per model family
CV_FOLDS = 5

SCOPE_COLUMN = "scope_urls"
OFFLINE_COLUMN = "persistence"        # 1 = offline / refresh token requested
TARGET_COLUMN = "risk_label"
ID_COLUMN = "combination_id"

RISK_ORDER = ["Low", "Medium", "High", "Critical"]

# ENCODING B: raw scopes + offline only. Flip to True to reproduce encoding C's
# feature matrix (adds one binary column per Google service); left False so B
# stays a distinct encoding from the finalized pipeline.
INCLUDE_SERVICE_FEATURES = False

# Carried through for traceability (not model inputs)
META_COLUMNS = ["combination_id", "source_category", "source_label", "scope_urls"]

# Engineered rubric features encoding B deliberately does NOT give the model.
# They are written to a reference file for cross-checking, and used by the
# dataset-level diagnostic below.
ENGINEERED_FEATURES = [
    "data_sensitivity", "access_level", "persistence", "transitive_exposure",
    "scope_count", "cross_service_breadth", "has_restricted_scope",
]

# Three-way split proportions (train is whatever remains: 0.70)
VAL_SIZE = 0.15
TEST_SIZE = 0.15

INPUT_PATH = "../dataset_cleaning/outputs/scope_guard_dataset_cleaned.csv"
SERVICE_CATALOG_PATH = "../dataset_creation/outputs/scopes_by_service.json"

OUTPUT_DIR = Path("b_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENCODED_DATASET_PATH = OUTPUT_DIR / "scope_guard_dataset_encodingB.csv"
VOCAB_PATH = OUTPUT_DIR / "scope_vocabulary.json"
REFERENCE_PATH = OUTPUT_DIR / "encodingB_engineered_reference.csv"

SCOPE_BINARIZER_PATH = OUTPUT_DIR / "scope_binarizer.joblib"
SERVICE_BINARIZER_PATH = OUTPUT_DIR / "service_binarizer.joblib"
MODEL_PATH = OUTPUT_DIR / "best_model.joblib"
FEATURE_NAMES_PATH = OUTPUT_DIR / "feature_names.json"
X_TRAIN_PATH = OUTPUT_DIR / "X_train.csv"
Y_TRAIN_PATH = OUTPUT_DIR / "y_train.csv"
BEST_PARAMS_PATH = OUTPUT_DIR / "best_params.json"
CV_RESULTS_PATH = OUTPUT_DIR / "cross_validation_results.csv"
RUN_MANIFEST_PATH = OUTPUT_DIR / "run_manifest.json"

CONFUSION_MATRIX_PATH = OUTPUT_DIR / "confusion_matrix.png"
MODEL_PERFORMANCE_METRICS_PATH = OUTPUT_DIR / "model_performance_metrics.png"
BEST_SHAP_SCOPE_EXPLANATION_PATH = OUTPUT_DIR / "best_shap_scope_explanation.png"
MEDIUM_SHAP_PATH = OUTPUT_DIR / "medium_shap.png"
LABEL_DIST_PATH = OUTPUT_DIR / "label_dist.png"
Y_TEST_DIST_PATH = OUTPUT_DIR / "y_test_distribution.png"
TUNING_COMPARISON_PATH = OUTPUT_DIR / "XGB_beforevafter_tuning.png"

SCORING = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_macro",
}

print(f"Artifacts will be written to: {OUTPUT_DIR.resolve()}")


## Shared helpers


In [ ]:
def seed_everything(seed=RANDOM_SEED):
    """Reset every global RNG this notebook touches.

    Called before each stochastic section so that re-running a single cell
    gives the same answer as running the notebook top to bottom.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def resolve_path(relative_path):
    """Find an input file whether the notebook runs from the repo or Colab.

    Tries the configured relative path first, then the bare filename in the
    working directory, then /content. Raises rather than silently returning a
    path that does not exist.
    """
    name = Path(relative_path).name
    for candidate in (Path(relative_path), Path.cwd() / name, Path("/content") / name):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {name!r}. Looked in: {relative_path}, "
        f"{Path.cwd() / name}, /content/{name}"
    )


def split_scopes(cell):
    """Split a '; '-separated scope_urls cell into a clean list of scope URLs."""
    return [s.strip() for s in str(cell).split(";") if s.strip()]


def short_name(scope_url):
    """
    Turn a full scope URL into a short, readable name for columns/plots.
        https://www.googleapis.com/auth/gmail.send  ->  gmail.send
    Bare OIDC scopes (openid, profile, email) are returned unchanged.
    """
    url = scope_url.strip()
    if "/" not in url:
        return url
    name = url.rstrip("/").split("/")[-1]
    return name if name else url


def plot_y_distribution(y_data, title, risk_order=RISK_ORDER, save_path=None):
    """Bar chart of label counts, ordered by risk_order.

    Accepts string labels or the integer codes the model is trained on.
    """
    if y_data is None or len(y_data) == 0:
        print(f"No data to plot for {title}")
        return

    series = pd.Series(list(y_data))
    if pd.api.types.is_numeric_dtype(series):
        series = series.map(lambda i: risk_order[int(i)])
    distribution = series.value_counts().reindex(risk_order).fillna(0)

    fig = plt.figure(figsize=(10, 7))
    sns.barplot(x=distribution.index, y=distribution.values, hue=distribution.index,
                palette="viridis", legend=False)
    plt.title(title, fontsize=20)
    plt.xlabel("Risk Label", fontsize=16)
    plt.ylabel("Count", fontsize=16)
    plt.xticks(fontsize=14, rotation=45, ha="right")
    plt.yticks(fontsize=14)
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    plt.close(fig)


def make_explainer(model, background):
    """Pick the right SHAP explainer from the model's *type*.

    Keying on the class rather than on a display string means renaming a
    candidate model can never silently downgrade a tree model to the slow
    model-agnostic explainer.
    """
    if isinstance(model, (XGBClassifier, RandomForestClassifier)):
        return shap.TreeExplainer(model)
    if isinstance(model, LogisticRegression):
        return shap.LinearExplainer(model, background)
    return shap.Explainer(model, background)


def class_shap_values(explainer, row_array, class_index):
    """Return the SHAP vector for one row and one class, shape-agnostic.

    SHAP returns a list (one array per class), a 3-D array, or a 2-D array
    depending on the model and version, so all three are handled here.
    """
    values = explainer.shap_values(row_array)
    if isinstance(values, list):
        return np.asarray(values[class_index][0])
    values = np.asarray(values)
    if values.ndim == 3:
        return values[0, :, class_index]
    return values[0]


seed_everything()


---
## Part 1 — Build the Encoding B dataset

Converts the cleaned dataset into the raw-scope representation: one binary
column per scope in the vocabulary (via `MultiLabelBinarizer`), plus `offline`
and the `risk_label` target. Nothing is invented — every binary column is
derived from the `scope_urls` already present in each row.

The seven engineered features are dropped from the model inputs (that is the
whole point of encoding B) but kept in a separate reference file for
cross-checking.


### 1.1 — Load the raw rows


In [ ]:
input_path = resolve_path(INPUT_PATH)
raw_df = pd.read_csv(input_path)
print(f"Loaded {len(raw_df)} rows from {input_path}")

scope_lists = raw_df[SCOPE_COLUMN].apply(split_scopes)
raw_df[[ID_COLUMN, SCOPE_COLUMN]].head()


### 1.2 — Binarize scopes with `MultiLabelBinarizer`

`MultiLabelBinarizer` takes the list of scopes for each row and produces one
binary column per distinct scope in the vocabulary — that is the entire
multi-hot encoding step. `classes_` is sorted, so column order is fixed.


In [ ]:
scope_binarizer = MultiLabelBinarizer()
scope_matrix = scope_binarizer.fit_transform(scope_lists)
scope_columns = [short_name(s) for s in scope_binarizer.classes_]

# Two different scope URLs could in principle collapse to the same short name
duplicate_names = sorted({c for c in scope_columns if scope_columns.count(c) > 1})
if duplicate_names:
    raise ValueError(f"Column-name collision(s) after shortening: {duplicate_names}")

print(f"Vocabulary: {len(scope_binarizer.classes_)} distinct scopes -> {len(scope_columns)} columns")


### 1.3 — Assemble the Encoding B dataframe


In [ ]:
encodingB_df = raw_df[META_COLUMNS].copy()
encodingB_df[scope_columns] = scope_matrix
encodingB_df["offline"] = (raw_df[OFFLINE_COLUMN].astype(str).str.strip() == "1").astype(int)
encodingB_df[TARGET_COLUMN] = raw_df[TARGET_COLUMN]

encodingB_df.head()


### 1.4 — Verify the encoding is faithful

Sanity checks: row count preserved, the number of 1s per row matches the
number of scopes in the original row, the `offline` flag matches the original
`persistence` column, and labels are unchanged.


In [ ]:
expected_scope_counts = scope_lists.apply(len)
actual_scope_counts = encodingB_df[scope_columns].sum(axis=1)
scope_mismatches = int((expected_scope_counts.values != actual_scope_counts.values).sum())

expected_offline = (raw_df[OFFLINE_COLUMN].astype(str).str.strip() == "1").astype(int)
offline_mismatches = int((encodingB_df["offline"] != expected_offline).sum())

label_mismatches = int((encodingB_df[TARGET_COLUMN] != raw_df[TARGET_COLUMN]).sum())

assert len(raw_df) == len(encodingB_df), "Row count changed!"

if scope_mismatches or offline_mismatches or label_mismatches:
    raise AssertionError(
        f"{scope_mismatches} scope-count mismatches, "
        f"{offline_mismatches} offline mismatches, {label_mismatches} label mismatches"
    )
print("All checks passed: scope counts, offline flags, and labels all match.")


### 1.5 — Write the encoded dataset


In [ ]:
encodingB_df.to_csv(ENCODED_DATASET_PATH, index=False)
print(f"Wrote {ENCODED_DATASET_PATH}  ({encodingB_df.shape[0]} rows x {encodingB_df.shape[1]} columns)")

vocab_map = dict(zip(scope_columns, scope_binarizer.classes_))
with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "n_scopes": len(vocab_map),
            "note": "Maps each binary column name back to its full OAuth scope URL.",
            "columns": vocab_map,
        },
        f, indent=2, sort_keys=True,
    )
print(f"Wrote {VOCAB_PATH}  ({len(vocab_map)} column -> URL mappings)")

reference_cols = [ID_COLUMN] + ENGINEERED_FEATURES + [TARGET_COLUMN]
raw_df[reference_cols].to_csv(REFERENCE_PATH, index=False)
print(f"Wrote {REFERENCE_PATH}  (engineered features kept for reference only)")


### 1.6 — Summary statistics


In [ ]:
total_cells = scope_matrix.size
sparsity = 100.0 * (1 - scope_matrix.sum() / total_cells)
per_row = scope_matrix.sum(axis=1)
offline_on = int(encodingB_df["offline"].sum())
label_counts = encodingB_df[TARGET_COLUMN].value_counts()

print("=" * 58)
print("ENCODING B DATASET SUMMARY")
print("=" * 58)
print(f"  Rows:                {len(encodingB_df)}")
print(f"  Scope columns:       {len(scope_columns)}")
print(f"  Model input columns: {len(scope_columns) + 1}  (scopes + offline)")
print(f"  Sparsity:            {sparsity:.1f}% zeros")
print(f"  Scopes per row:      min={per_row.min()}, max={per_row.max()}, mean={per_row.mean():.2f}")
print(f"  Offline requested:   {offline_on} rows ({offline_on / len(encodingB_df) * 100:.1f}%)")

print("\n  Label distribution:")
for tier in RISK_ORDER:
    n = int(label_counts.get(tier, 0))
    print(f"    {tier:>8}: {n:>5} ({n / len(encodingB_df) * 100:.1f}%)")

scope_counts = pd.Series(scope_matrix.sum(axis=0), index=scope_columns).sort_values(ascending=False)
print("\n  Most frequently requested scopes:")
for name, n in scope_counts.head(8).items():
    print(f"    {name:>34}: {int(n)}")

rare = scope_counts[scope_counts <= 2]
print(f"\n  Scopes appearing in <=2 rows: {len(rare)} (these will be hard for the model to learn)")


---
## Part 2 — Train and explain the Encoding B model


### 2.1 — Load the dataset and encode the target


In [ ]:
df = pd.read_csv(input_path)
print(f"Loaded {len(df)} rows from {input_path}")

# OrdinalEncoder needs a 2D, single-column input plus an explicit category
# order, so it pins Low=0, Medium=1, High=2, Critical=3 exactly (rather than
# picking up whatever order the labels happen to appear in the data).
label_encoder = OrdinalEncoder(categories=[RISK_ORDER], dtype=int)
y = label_encoder.fit_transform(df[[TARGET_COLUMN]]).ravel().astype(int)

plot_y_distribution(y, 'Risk Distribution of "y"')

print("Label distribution:")
for tier in RISK_ORDER:
    n = int((df[TARGET_COLUMN] == tier).sum())
    print(f"  {tier}: {n} ({n / len(df) * 100:.1f}%)")


### 2.2 — Build the Encoding B feature matrix

One `MultiLabelBinarizer` over individual scopes plus the `offline` flag —
that is the whole feature matrix. Column order is the binarizer's sorted
`classes_` order followed by `offline`, so the matrix is identical on any
machine.

If `INCLUDE_SERVICE_FEATURES` is switched on, a second binarizer over the
services those scopes belong to is stacked in between, which reproduces
encoding C.


In [ ]:
scope_lists = df[SCOPE_COLUMN].apply(split_scopes)

scope_binarizer = MultiLabelBinarizer()
scope_matrix = scope_binarizer.fit_transform(scope_lists)
scope_feature_names = [short_name(s) for s in scope_binarizer.classes_]
print(f"Scope vocabulary: {len(scope_binarizer.classes_)} distinct scopes")

blocks = [scope_matrix]
feature_names = list(scope_feature_names)
service_binarizer = None

if INCLUDE_SERVICE_FEATURES:
    with open(resolve_path(SERVICE_CATALOG_PATH), encoding="utf-8") as f:
        service_catalog = json.load(f)

    scope_to_service = {s.strip(): service
                        for service, scopes in service_catalog.items() for s in scopes}
    services = sorted(service_catalog.keys())
    print(f"{len(services)} services covering {len(scope_to_service)} scope URLs")

    unmapped = sorted({s for scopes in scope_lists for s in scopes if s not in scope_to_service})
    if unmapped:
        print(f"  WARNING: {len(unmapped)} scopes not found in the service catalog "
              f"(no service flag set for them): {unmapped}")

    service_lists = scope_lists.apply(
        lambda scopes: [scope_to_service[s] for s in scopes if s in scope_to_service])
    service_binarizer = MultiLabelBinarizer(classes=services)
    service_matrix = service_binarizer.fit_transform(service_lists)

    blocks.append(service_matrix)
    feature_names += [f"svc:{s}" for s in services]

# offline flag: last column, always
offline_flag = (df[OFFLINE_COLUMN].astype(str).str.strip() == "1").astype(np.float32).values.reshape(-1, 1)
blocks.append(offline_flag)
feature_names.append("offline")

X = np.hstack(blocks).astype(np.float32)

print(f"\nEncoded matrix: {X.shape[0]} rows x {X.shape[1]} columns ({(X == 0).mean() * 100:.1f}% zeros)")
print(f"  = {len(scope_feature_names)} scope cols"
      + (f" + {len(feature_names) - len(scope_feature_names) - 1} service cols" if INCLUDE_SERVICE_FEATURES else "")
      + " + offline")
assert X.shape[1] == len(feature_names)


### 2.3 — Dataset-level diagnostic: engineered vs. raw features

Descriptive only — it runs on the **full** dataset and selects no model. It
contrasts the engineered encoding (a near-lookup) with the raw-scope encoding
(a genuine learning task).


In [ ]:
def compare_encodings(source, X_raw, y):
    eng_vectors = source[ENGINEERED_FEATURES].apply(tuple, axis=1).nunique()
    scope_sets = source[SCOPE_COLUMN].apply(lambda c: tuple(sorted(split_scopes(c)))).nunique()
    print(f"Unique engineered-feature vectors: {eng_vectors} / {len(source)} rows")
    print(f"Unique scope-sets:                 {scope_sets} / {len(source)} rows")
    print("(Few unique feature vectors => dense => lookup.\n"
          " Many unique scope-sets => sparse => genuine prediction.)")

    X_eng = source[ENGINEERED_FEATURES].values.astype(float)
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    diag_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                               random_state=RANDOM_SEED, eval_metric="mlogloss",
                               tree_method="hist", nthread=1)

    f1_eng = cross_validate(diag_model, X_eng, y, cv=cv, scoring="f1_macro", n_jobs=1)["test_score"].mean()
    f1_raw = cross_validate(diag_model, X_raw, y, cv=cv, scoring="f1_macro", n_jobs=1)["test_score"].mean()
    print(f"\nXGBoost macro-F1, engineered features: {f1_eng:.3f}  (near-ceiling => lookup)")
    print(f"XGBoost macro-F1, encoding B:          {f1_raw:.3f}  (real learning problem)")


seed_everything()
compare_encodings(df, X, y)


### 2.4 — Three-way split (70 / 15 / 15, stratified)

Split first, so model selection can never see the validation or test rows.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=(VAL_SIZE + TEST_SIZE), stratify=y, random_state=RANDOM_SEED,
)
relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=relative_test_size, stratify=y_temp, random_state=RANDOM_SEED,
)

total = len(X)
print(f"{'Set':<12}{'Rows':>7}{'%':>8}   " + "".join(f"{t:>10}" for t in RISK_ORDER))
for name, yy in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    counts = [int((yy == i).sum()) for i in range(len(RISK_ORDER))]
    print(f"{name:<12}{len(yy):>7}{len(yy) / total * 100:>7.1f}%   " + "".join(f"{n:>10}" for n in counts))

# A cheap content fingerprint of each split: if these change, the split changed.
split_fingerprint = {
    name: int(pd.util.hash_array(np.ascontiguousarray(arr)).sum() % (2 ** 63))
    for name, arr in [("train", X_train), ("val", X_val), ("test", X_test)]
}
print("\nSplit fingerprint (identical across machines):")
for name, h in split_fingerprint.items():
    print(f"  {name:>5}: {h}")

print("\nThe TEST set is now locked. It is not touched again until section 2.7.")


### 2.5 — Hyperparameter tuning

Every model family is fit on the whole training split, scored on the
validation split, and tuned for 100 Optuna trials. The best-scoring trial
(by macro F1) for each family goes to the next stage.

Each study gets its own `TPESampler(seed=RANDOM_SEED)` and runs with
`n_jobs=1`: parallel trials complete in nondeterministic order, which changes
what the TPE sampler has seen when it proposes the next trial, and that alone
would make the search unreproducible.


In [ ]:
seed_everything()

def objective(trial):
    params = {
        "objective": "multi:softprob",
        "num_class": len(RISK_ORDER),
        "eval_metric": "mlogloss",     # XGBoost's internal validation tracking
        "booster": "gbtree",
        "tree_method": "hist",         # pinned: the default has changed between versions
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", .05, 0.5, log=False),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "random_state": RANDOM_SEED,
        "nthread": 1,
    }

    # Pruning keys off mlogloss internally — F1 isn't computed per-round by XGBoost
    pruning_callback = XGBoostPruningCallback(trial, "validation_0-mlogloss")

    # callbacks go in the constructor, not fit() — required since XGBoost 2.1.0
    model = XGBClassifier(**params, callbacks=[pruning_callback])
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    preds = model.predict(X_val)               # class labels, not probabilities
    return f1_score(y_val, preds, average="macro")


sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler,
                            pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)

print("Best params:", study.best_params)
print("Best value:", study.best_value)
xgb_best_params = copy.deepcopy(study.best_params)
xgb_best_params.update({"random_state": RANDOM_SEED, "nthread": 1,
                        "tree_method": "hist", "eval_metric": "mlogloss"})


In [ ]:
seed_everything()

def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_features": trial.suggest_int("max_features", 6, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20, log=False),
        "random_state": RANDOM_SEED,
        "n_jobs": 1,
    }

    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    return f1_score(y_val, preds, average="macro")


sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)

print("Best params:", study.best_params)
print("Best value:", study.best_value)
rf_best_params = copy.deepcopy(study.best_params)
rf_best_params.update({"random_state": RANDOM_SEED, "n_jobs": 1})


In [ ]:
seed_everything()

def objective(trial):
    params = {
        "C": trial.suggest_float("C", .5, 3.0),
        "tol": trial.suggest_float("tol", 1e-5, 1e-3, log=True),
        "max_iter": trial.suggest_int("max_iter", 50, 250),
        "random_state": RANDOM_SEED,
        "solver": "lbfgs",   # pinned: the sklearn default is version-dependent
    }

    model = LogisticRegression(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    return f1_score(y_val, preds, average="macro")


sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)

print("Best params:", study.best_params)
print("Best value:", study.best_value)
logreg_best_params = copy.deepcopy(study.best_params)
logreg_best_params.update({"random_state": RANDOM_SEED, "solver": "lbfgs"})

with open(BEST_PARAMS_PATH, "w", encoding="utf-8") as f:
    json.dump({"XGBoost": xgb_best_params,
               "RandomForest": rf_best_params,
               "LogisticRegression": logreg_best_params},
              f, indent=2, sort_keys=True, default=str)
print(f"Saved: {BEST_PARAMS_PATH}")


### 2.6 — Tuned model selection

Each family is re-instantiated with its best trial's hyperparameters and fit on
the training split under 5-fold stratified CV. The family with the highest mean
macro F1 is selected.


In [ ]:
seed_everything()

candidate_models = {
    "XGBoost": XGBClassifier(**xgb_best_params),
    "RandomForest": RandomForestClassifier(**rf_best_params),
    "LogisticRegression": LogisticRegression(**logreg_best_params),
}
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

cv_results = {}
for name, candidate in candidate_models.items():
    scores = cross_validate(candidate, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=1)
    cv_results[name] = {metric: scores[f"test_{metric}"] for metric in SCORING}

    print(f"\n{name}")
    for fold in range(CV_FOLDS):
        fold_scores = "  ".join(f"{m}={cv_results[name][m][fold]:.4f}" for m in SCORING)
        print(f"  Fold {fold}: {fold_scores}")
    means = "  ".join(f"{m}={cv_results[name][m].mean():.4f}" for m in SCORING)
    print(f"  MEAN:   {means}")

cv_summary = pd.DataFrame({
    name: {f"{metric}_mean": scores[metric].mean() for metric in SCORING}
          | {f"{metric}_std": scores[metric].std() for metric in SCORING}
    for name, scores in cv_results.items()
}).T[[f"{m}_{s}" for m in SCORING for s in ("mean", "std")]]
cv_summary.to_csv(CV_RESULTS_PATH)
print(f"\nSaved: {CV_RESULTS_PATH}")

# Ties are broken by name so the selection cannot depend on dict ordering.
ranking = sorted(cv_results, key=lambda n: (-cv_results[n]["f1"].mean(), n))
best_name = ranking[0]

print(f"\nSELECTED MODEL:  {best_name} "
      f"(CV macro-F1 = {cv_results[best_name]['f1'].mean():.4f} "
      f"+/- {cv_results[best_name]['f1'].std():.4f})")
best_model = candidate_models[best_name]
print(best_model)


### 2.6b — Fit the selected model on the full training split


In [ ]:
seed_everything()
best_model.fit(X_train, y_train)

print(best_name)
best_y_pred_val = best_model.predict(X_val)
print(f"Accuracy: {accuracy_score(y_val, best_y_pred_val):.3f}")
print(f"Macro-F1: {f1_score(y_val, best_y_pred_val, average='macro'):.3f}\n")
print(classification_report(y_val, best_y_pred_val, target_names=RISK_ORDER, zero_division=0))


### 2.7 — Final test evaluation

Touched exactly once, on the single already-committed model. These are the
numbers to report.


In [ ]:
best_y_pred_test = best_model.predict(X_test)

print(best_name)
print(f"Accuracy: {accuracy_score(y_test, best_y_pred_test):.4f}")
print(f"Macro-F1: {f1_score(y_test, best_y_pred_test, average='macro'):.4f}\n")
print(classification_report(y_test, best_y_pred_test, target_names=RISK_ORDER, zero_division=0))


### 2.8 — Pick one test row to explain

Drawn from a seeded `Generator`, so the explained row is the same one on every
machine.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
sample_idx = int(rng.integers(0, len(X_test)))
sample = X_test[sample_idx]

best_predicted_class = int(best_model.predict(X_test[sample_idx:sample_idx + 1])[0])
best_predicted_label = RISK_ORDER[best_predicted_class]

scopes_present = [feature_names[i] for i in range(len(feature_names)) if sample[i] == 1.0]

print(f"Test row #{sample_idx}")
print(f"True label:      {RISK_ORDER[int(y_test[sample_idx])]}")
print(f"Scopes present:  {', '.join(scopes_present)}")
print(f"{best_name} model prediction: {best_predicted_label}")


### 2.9 — Local SHAP explanation

Which scopes drove this particular prediction, for the top 10 by absolute
SHAP value.


In [ ]:
top_k = 10

explainer = make_explainer(best_model, X_train)
shap_vals = class_shap_values(explainer, X_test[sample_idx:sample_idx + 1], best_predicted_class)

ranked = sorted(zip(feature_names, shap_vals, X_test[sample_idx]), key=lambda t: -abs(t[1]))[:top_k]

print(f"Top {top_k} scope contributions toward '{best_predicted_label}' for {best_name}:")
for name, val, present in ranked:
    arrow = "increases" if val > 0 else "decreases"
    tag = "present" if present == 1.0 else "absent"
    print(f"  {name:>32} ({tag:>7}): {val:+.3f} ({arrow} model confidence)")


### 2.10 — Plot the SHAP explanation


In [ ]:
# Poster sizing: 10 horizontal bars with long labels at 32pt need a wide
# canvas, otherwise the y-labels eat the plotting area and squash the bars.
POSTER_FONTSIZE = 32
fig, ax = plt.subplots(figsize=(20, 9))

# `ranked` is already sorted by descending absolute SHAP value; seaborn draws
# the first category at the top, so passing it as-is puts the largest
# magnitude at the top regardless of sign.
names = [f"{t[0]} ({'present' if t[2] == 1.0 else 'absent'})" for t in ranked]
scores = [float(t[1]) for t in ranked]

viridis = sns.color_palette("viridis", n_colors=10)
pos_color, neg_color = viridis[7], viridis[1]
colors = [pos_color if s > 0 else neg_color for s in scores]

sns.barplot(x=scores, y=names, hue=names, palette=colors, dodge=False, legend=False, ax=ax, orient="h")
ax.set_yticks(np.arange(len(names)))
ax.set_yticklabels(names, fontsize=POSTER_FONTSIZE)
ax.tick_params(axis="x", labelsize=POSTER_FONTSIZE)
ax.axvline(0, color="#333", linewidth=2.0)
ax.set_xlabel("SHAP value", fontsize=POSTER_FONTSIZE, labelpad=14)
ax.set_title(f"SHAP — scopes driving '{best_predicted_label}' for {best_name}",
             fontsize=POSTER_FONTSIZE, pad=22)
ax.grid(axis="x", linestyle="--", alpha=0.7, color="lightgrey")
for spine in ax.spines.values():
    spine.set_linewidth(1.5)

plt.tight_layout()
# dpi 300 for print; 150 looks soft when blown up to poster scale.
plt.savefig(BEST_SHAP_SCOPE_EXPLANATION_PATH, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved: {BEST_SHAP_SCOPE_EXPLANATION_PATH}")


### 2.10b — SHAP explanation for a Medium-risk combination


In [ ]:
# A row guaranteed to be Medium risk, chosen from a seeded Generator.
medium_positions = np.where(y_test == RISK_ORDER.index("Medium"))[0]

if len(medium_positions) == 0:
    print("No Medium-risk rows in the test split; skipping this explanation.")
else:
    rng = np.random.default_rng(RANDOM_SEED)
    sample_idx = int(rng.choice(medium_positions))
    sample = X_test[sample_idx]

    best_predicted_class = int(best_model.predict(X_test[sample_idx:sample_idx + 1])[0])
    best_predicted_label = RISK_ORDER[best_predicted_class]

    scopes_present = [feature_names[i] for i in range(len(feature_names)) if sample[i] == 1.0]

    print(f"Test row #{sample_idx}")
    print(f"True label:      {RISK_ORDER[int(y_test[sample_idx])]}")
    print(f"Scopes present:  {', '.join(scopes_present)}")
    print(f"{best_name} model prediction: {best_predicted_label}")

    top_k = 10
    explainer = make_explainer(best_model, X_train)
    shap_vals = class_shap_values(explainer, X_test[sample_idx:sample_idx + 1], best_predicted_class)
    ranked = sorted(zip(feature_names, shap_vals, X_test[sample_idx]), key=lambda t: -abs(t[1]))[:top_k]

    print(f"\nTop {top_k} scope contributions toward '{best_predicted_label}' for {best_name}:")
    for name, val, present in ranked:
        arrow = "increases" if val > 0 else "decreases"
        tag = "present" if present == 1.0 else "absent"
        print(f"  {name:>32} ({tag:>7}): {val:+.3f} ({arrow} model confidence)")


In [ ]:
POSTER_FONTSIZE = 32
fig, ax = plt.subplots(figsize=(20, 9))

names = [f"{t[0]} ({'present' if t[2] == 1.0 else 'absent'})" for t in ranked]
scores = [float(t[1]) for t in ranked]

viridis = sns.color_palette("viridis", n_colors=10)
pos_color, neg_color = viridis[7], viridis[1]
colors = [pos_color if s > 0 else neg_color for s in scores]

sns.barplot(x=scores, y=names, hue=names, palette=colors, dodge=False, legend=False, ax=ax, orient="h")
ax.set_yticks(np.arange(len(names)))
ax.set_yticklabels(names, fontsize=POSTER_FONTSIZE)
ax.tick_params(axis="x", labelsize=POSTER_FONTSIZE)
ax.axvline(0, color="#333", linewidth=2.0)
ax.set_xlabel("SHAP value", fontsize=POSTER_FONTSIZE, labelpad=14)
ax.set_title(f"SHAP — scopes driving '{best_predicted_label}' for {best_name}",
             fontsize=POSTER_FONTSIZE, pad=22)
ax.grid(axis="x", linestyle="--", alpha=0.7, color="lightgrey")
for spine in ax.spines.values():
    spine.set_linewidth(1.5)

plt.tight_layout()
plt.savefig(MEDIUM_SHAP_PATH, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved: {MEDIUM_SHAP_PATH}")


### 2.11 — Model performance metrics


In [ ]:
accuracy = accuracy_score(y_test, best_y_pred_test)
f1_macro = f1_score(y_test, best_y_pred_test, average="macro", zero_division=0)
precision_macro = precision_score(y_test, best_y_pred_test, average="macro", zero_division=0)
recall_macro = recall_score(y_test, best_y_pred_test, average="macro", zero_division=0)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Macro F1", "Macro Precision", "Macro Recall"],
    "Score": [accuracy, f1_macro, precision_macro, recall_macro],
})

POSTER_FONTSIZE = 32
fig, ax = plt.subplots(figsize=(20, 13))
sns.barplot(x="Metric", y="Score", data=metrics_df, hue="Metric", palette="viridis",
            dodge=False, legend=False, ax=ax)

ax.set_title(f"{best_name} Performance on Test Set (Encoding B)", fontsize=POSTER_FONTSIZE, pad=22)
ax.set_xlabel("Metric", fontsize=POSTER_FONTSIZE, labelpad=14)
ax.set_ylabel("Score", fontsize=POSTER_FONTSIZE, labelpad=14)
ax.set_ylim(0, 1.08)  # headroom so the value labels aren't clipped
ax.tick_params(axis="x", labelsize=POSTER_FONTSIZE)
ax.tick_params(axis="y", labelsize=POSTER_FONTSIZE)
ax.grid(axis="y", linestyle="--", alpha=0.7, color="lightgrey")
for spine in ax.spines.values():
    spine.set_linewidth(1.5)

for i, row in metrics_df.iterrows():
    ax.text(i, row["Score"] + 0.02, f"{row['Score']:.3f}", ha="center", va="bottom",
            fontsize=POSTER_FONTSIZE)

plt.tight_layout()
plt.savefig(MODEL_PERFORMANCE_METRICS_PATH, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved: {MODEL_PERFORMANCE_METRICS_PATH}")
metrics_df


### 2.12 — Confusion matrix


In [ ]:
cm = confusion_matrix(y_test, best_y_pred_test, labels=np.arange(len(RISK_ORDER)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=RISK_ORDER)

fig, ax = plt.subplots(figsize=(12, 12))
disp.plot(cmap=plt.cm.Blues, ax=ax, values_format="d")

ax.set_title("Confusion Matrix (Test Set)", fontsize=32)
ax.set_xlabel("Predicted Label", fontsize=32)
ax.set_ylabel("True Label", fontsize=32)
ax.tick_params(axis="x", labelsize=24, rotation=45)
ax.tick_params(axis="y", labelsize=24, rotation=0)
for labels in disp.text_:
    for label in labels:
        label.set_fontsize(20)

plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved: {CONFUSION_MATRIX_PATH}")


### 2.13 — True label distribution in the test set


In [ ]:
plot_y_distribution(y_test, "Distribution of True Risk Labels in Test Set",
                    save_path=Y_TEST_DIST_PATH)


### 2.13b — Risk label distribution across the full dataset


In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(
    data=df,
    x=TARGET_COLUMN,
    order=RISK_ORDER,
    hue=TARGET_COLUMN,
    palette="viridis",
    legend=False,
)
plt.title("Distribution of Risk Labels", fontsize=24)
plt.xlabel("Risk Label", fontsize=22)
plt.ylabel("Count", fontsize=22)
plt.xticks(rotation=45, ha="right", fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.savefig(LABEL_DIST_PATH, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {LABEL_DIST_PATH}")


### 2.13c — XGBoost before vs. after tuning

An untuned XGBoost with library defaults, scored on the validation split
against the tuned model. This is the figure that shows what the 100 Optuna
trials actually bought.


In [ ]:
def val_metrics(model, X, y):
    pred = model.predict(X)
    return {
        "Accuracy":  accuracy_score(y, pred),
        "F1":        f1_score(y, pred, average="macro", zero_division=0),
        "Precision": precision_score(y, pred, average="macro", zero_division=0),
        "Recall":    recall_score(y, pred, average="macro", zero_division=0),
    }


seed_everything()
old_xgb = XGBClassifier(random_state=RANDOM_SEED, nthread=1, tree_method="hist",
                        eval_metric="mlogloss")
old_xgb.fit(X_train, y_train)

tuned_xgb = XGBClassifier(**xgb_best_params)
tuned_xgb.fit(X_train, y_train)

baseline = val_metrics(old_xgb, X_val, y_val)
tuned = val_metrics(tuned_xgb, X_val, y_val)

comparison_df = pd.DataFrame(
    [{"Metric": m, "Model": "Baseline", "Score": v} for m, v in baseline.items()]
    + [{"Metric": m, "Model": "Tuned", "Score": v} for m, v in tuned.items()]
)

VIRIDIS_PURPLE = "#494c7f"   # dark end of viridis
VIRIDIS_GREEN  = "#80c161"   # light green end

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=comparison_df, x="Metric", y="Score", hue="Model",
    palette={"Baseline": VIRIDIS_PURPLE, "Tuned": VIRIDIS_GREEN},
    edgecolor="white", linewidth=0.8,
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=9)

ax.set_ylim(0, 1.05)
ax.set_ylabel("Score (macro-averaged)", fontsize=16)
ax.set_xlabel("")
ax.set_title("XGBoost Validation Performance: Before vs. After Hyperparameter Tuning", fontsize=13)
ax.legend(title="", frameon=False, fontsize=16)
plt.yticks(fontsize=14)
plt.xticks(fontsize=16)
sns.despine()
plt.tight_layout()
plt.savefig(TUNING_COMPARISON_PATH, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {TUNING_COMPARISON_PATH}")


### 2.14 — Save the model and supporting artifacts

Everything lands in `b_outputs/`, including a run manifest that records the
seed, the library versions, the split fingerprint, and the selected model — so
a later run can be checked against this one rather than merely re-run.


In [ ]:
joblib.dump(best_model, MODEL_PATH)
print(f"Trained model saved to {MODEL_PATH}")

joblib.dump(scope_binarizer, SCOPE_BINARIZER_PATH)
print(f"Scope binarizer saved to {SCOPE_BINARIZER_PATH}")

if service_binarizer is not None:
    joblib.dump(service_binarizer, SERVICE_BINARIZER_PATH)
    print(f"Service binarizer saved to {SERVICE_BINARIZER_PATH}")

with open(FEATURE_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump({"encoding": "B: raw scope columns + offline flag",
               "include_service_features": bool(INCLUDE_SERVICE_FEATURES),
               "n_features": len(feature_names),
               "feature_order": feature_names},
              f, indent=2)
print(f"Feature names saved to {FEATURE_NAMES_PATH}")

pd.DataFrame(X_train, columns=feature_names).to_csv(X_TRAIN_PATH, index=False)
pd.DataFrame(y_train, columns=[TARGET_COLUMN]).to_csv(Y_TRAIN_PATH, index=False)
print(f"Saved {X_TRAIN_PATH} and {Y_TRAIN_PATH}")

manifest = {
    "encoding": "B",
    "include_service_features": bool(INCLUDE_SERVICE_FEATURES),
    "random_seed": RANDOM_SEED,
    "n_optuna_trials": N_TRIALS,
    "cv_folds": CV_FOLDS,
    "input_path": str(input_path),
    "n_rows": int(len(df)),
    "n_features": int(len(feature_names)),
    "split_sizes": {"train": int(len(X_train)), "val": int(len(X_val)), "test": int(len(X_test))},
    "split_fingerprint": split_fingerprint,
    "selected_model": best_name,
    "best_params": {"XGBoost": xgb_best_params,
                    "RandomForest": rf_best_params,
                    "LogisticRegression": logreg_best_params},
    "test_metrics": {"accuracy": float(accuracy), "macro_f1": float(f1_macro),
                     "macro_precision": float(precision_macro), "macro_recall": float(recall_macro)},
    "environment": ENVIRONMENT,
}
with open(RUN_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, sort_keys=True, default=str)
print(f"Run manifest saved to {RUN_MANIFEST_PATH}")
